In [1]:
# This is the code to merge all MTL beta csv

import pandas as pd
import glob

# === Define list of TSV files ===
# Adjust the pattern to match your actual filenames
tsv_files = glob.glob("subj*_MTL_trial_betas.tsv")

# === Load and concatenate ===
all_dfs = [pd.read_csv(f, sep="\t") for f in tsv_files]
combined_df = pd.concat(all_dfs, ignore_index=True)

# === Save combined TSV ===
combined_df.to_csv("all_subjects_MTL_trial_betas.tsv", sep="\t", index=False)

print(f"✅ Combined {len(tsv_files)} files into 'all_subjects_MTL_trial_betas.tsv'")
print(combined_df.head())

✅ Combined 3 files into 'all_subjects_MTL_trial_betas.tsv'
  Subject    Session  Trial  ROI_Beta
0  subj01  session01      0  0.134300
1  subj01  session01      1  0.033132
2  subj01  session01      2  0.248891
3  subj01  session01      3  0.071685
4  subj01  session01      4  0.097249


In [2]:
# and you can use this to merge with behavior :

import pandas as pd

# === Load the two dataframes ===
betas_df = pd.read_csv("all_subjects_MTL_trial_betas.tsv", sep="\t")
responses_df = pd.read_csv("all_subjects_responses.tsv", sep="\t")

# === Sanity check: Confirm same number of rows ===
assert len(betas_df) == len(responses_df), "❌ Row count mismatch between betas and responses"

# === Add the ROI_Beta column ===
responses_df["MTL_ROI_Beta"] = betas_df["ROI_Beta"].values

# === Save updated responses ===
responses_df.to_csv("all_subjects_responses_with_beta.tsv", sep="\t", index=False)

print(f"✅ New file saved with ROI_Beta column: {len(responses_df)} rows")

FileNotFoundError: [Errno 2] No such file or directory: 'all_subjects_responses.tsv'

In [ ]:
# This script is designed to extract trial-level average beta values from a region of interest (ROI) — specifically the MTL (medial temporal lobe) — for multiple subjects
# the person who made this also did a version for all subregions of MTL rather than the whole MTL

from cloudpathlib import CloudPath, S3Client, S3Path
import nibabel as nib
import pandas as pd

# Define s3_client first
s3_client = S3Client(no_sign_request=True)

nsd_base = CloudPath("s3://natural-scenes-dataset/", client=s3_client)

subjects = [f"subj0{i}" for i in range(1, 9)]  # subj01 to subj08

for subj in subjects:
    print(f"\n🔄 Processing {subj}...")

    # Define paths
    beta_dir = nsd_base / "nsddata_betas" / "ppdata" / subj / "func1pt8mm" / "betas_fithrf_GLMdenoise_RR"
    roi_path = nsd_base / "nsddata" / "ppdata" / subj / "func1pt8mm" / "roi" / "MTL.nii.gz"

    # Load ROI mask once
    roi_img = nib.load(roi_path.fspath)
    roi_mask = roi_img.get_fdata() > 0

    trial_list = []

    # Loop through all session files
    for beta_file in beta_dir.iterdir():
        if beta_file.name.startswith("betas_session") and beta_file.name.endswith(".nii.gz"):
            session = beta_file.name.replace("betas_", "").replace(".nii.gz", "")
            print(f"  🧠 Loading {session}...")

            beta_img = nib.load(beta_file.fspath)
            beta_data = beta_img.get_fdata()

            n_trials = beta_data.shape[-1]

            for i in range(n_trials):
                roi_beta = (beta_data[..., i][roi_mask] / 300).mean()
                trial_list.append({
                    "Subject": subj,
                    "Session": session,
                    "Trial": i,
                    "ROI_Beta": roi_beta
                })

    # Save to TSV
    df = pd.DataFrame(trial_list)
    out_path = f"{subj}_MTL_trial_betas.tsv"
    df.to_csv(out_path, sep="\t", index=False)
    print(f"✅ Saved {len(df)} trial beta values to {out_path}")